In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

In [2]:
import xgboost as xgb
import dill #aneto
import SuperLore
import category_encoders


## Data Loading

In [3]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")




In [4]:
bb


XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=0.65, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.025, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=13, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=110, n_jobs=10, num_parallel_tree=None,
              objective='binary:hinge', predictor=None, ...)

In [5]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')



In [6]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

In [7]:
for lore_p in objs[6].rule.premises:
    print(lore_p.op)
    

=
=


In [8]:
#carico shapley value full
path =('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_explanations_full.p')
explanations_shap = dill.load(open(path, 'rb'))

# Data preparation

In [9]:
feature_names = X_test.columns
real_feature_names = X_test.columns

In [10]:
inst = X_test.iloc[1].values
inst

array([0.52980757, 0.32495991, 0.45295295, 0.        , 0.        ,
       0.        , 0.75875876, 0.        , 0.52593287, 0.23923411,
       0.        , 0.25880995, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.58079442, 0.        , 0.61974726,
       0.46246246, 0.4984985 , 0.29654024, 1.        , 0.07107107,
       0.        , 0.65890488])

In [11]:
for lore_p in objs[1].rule.premises:
    if lore_p.att == f:
        op = lore_p.op
        thr = lore_p.thr
        is_continuous = lore_p.is_continuous
        temp[f]['op']= op
print(temp)

NameError: name 'f' is not defined

In [12]:
X_test.head(7).T

,0,1,2,3,4,5,6
PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO,0.000000,0.529808,0.523023,0.611950,0.556724,0.440114,0.541083
PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO,0.227975,0.324960,0.204011,0.291709,0.935637,0.487688,0.357585
PRODV_LETTERE_DI_CREDITO_PON,0.990218,0.452953,0.452953,0.452953,0.452953,0.452953,0.452953
SCADV_FLG_RATA_DIVISA_SEK,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
OWNER_PRODV_FOREX_PON,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM,0.947305,0.758759,0.233552,0.130130,0.357976,0.105060,0.937732
PN_LC_IMPORT_FLG_ONLY_TY,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PN_SEPA_ENTRATA_TY_VAL,0.921458,0.525933,0.026026,0.319746,0.341334,0.062555,0.495465
SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE,0.511439,0.239234,0.388234,0.311553,0.821015,0.246194,0.323923


In [13]:
temp={}
i=inst
for j,f in enumerate(feature_names):
        temp[f]= dict()
        temp[f]['feature_importance'] = explanations_shap[0][j]
print(temp)

{'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': {'feature_importance': 0.09466361575103292}, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': {'feature_importance': 0.08120621911794842}, 'PRODV_LETTERE_DI_CREDITO_PON': {'feature_importance': 0.2854290714347371}, 'SCADV_FLG_RATA_DIVISA_SEK': {'feature_importance': 0.0}, 'OWNER_PRODV_FOREX_PON': {'feature_importance': 0.3347166349758118}, 'OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON': {'feature_importance': 0.11593095222277043}, 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': {'feature_importance': 0.03719840600451789}, 'PN_LC_IMPORT_FLG_ONLY_TY': {'feature_importance': 0.0}, 'PN_SEPA_ENTRATA_TY_VAL': {'feature_importance': 0.1827153803166584}, 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': {'feature_importance': -0.02485413807482473}, 'OWNER_PRODV_GARANZIE_DOMESTICHE_PON': {'feature_importance': 0.010956627331615891}, 'PN_SEPA_USCITA_PRC_DLT_YEAR_VAL': {'feature_importance': -0.03174290419792669}, 'SCADV_FLG_RATA_DIVISA_AUD': {'feature_importance': 0.0}, 'OWNER_PRODV_AN

In [14]:
explanations_shap

array([[ 0.09466362,  0.08120622,  0.28542907, ..., -0.07626389,
         0.        , -0.02731891],
       [-0.02177902,  0.07350576, -0.01937944, ...,  0.0781945 ,
         0.        ,  0.96232297],
       [ 0.01969376,  0.08790127, -0.03470209, ..., -0.16089288,
         0.        , -0.26676439],
       ...,
       [ 0.02056669,  0.08696288, -0.03176322, ...,  0.13737915,
         0.        , -0.17642036],
       [-0.00200953,  0.06920489, -0.03739386, ..., -0.07819865,
         0.        , -0.19285882],
       [ 0.04856581, -0.03532057, -0.01439595, ..., -0.01933182,
         0.        ,  0.06414079]])

In [58]:
def data_to_plot(feature_names=feature_names, explanations_shap=explanations_shap, instance=0):
    temp={}
    i=instance
    l='[0.0, 1.0]'
    df_viz = pd.DataFrame(columns = ['type', 'name', 'rname', 'min', 'max', 'q1', 'median', 'q3', 'mean',
           'std', 'feature_importance', 'category', 'count', 'inst', 'op', 'thr',
           'is_continuous', 'thr2'])
    
    bool_f= False
    for j,f in enumerate(feature_names):
        temp[f]= dict()
        temp[f]['feature_importance'] = explanations_shap[0][j]
        for lore_p in objs[i].rule.premises:
            if lore_p.att == f:
                op = lore_p.op
                thr = lore_p.thr
                is_continuous = lore_p.is_continuous
                temp[f]['op']= op
                temp[f]['thr']= thr
                temp[f]['is_continuous']= is_continuous
                bool_f= True
        if len(np.unique(X_train[f]))==2:
            type_f = 'categorical'
            count=np.unique(X_train[f], return_counts= True)
            temp[f]['type']= type_f
            temp[f]['count']= count
        else:
            type_f = 'numeric'
            temp[f]['type']= type_f
            min_f = X_train[f].min()
            temp[f]['min']= min_f
            max_f = X_train[f].max()
            temp[f]['max']= max_f
            q_1 =X_train[f].quantile(0.25)
            temp[f]['q1']= q_1
            median =X_train[f].quantile(0.50)
            temp[f]['median']= median
            q_3=X_train[f].quantile(0.75)
            temp[f]['q3']= q_3
            temp[f]['mean'] = X_train[f].mean()
            temp[f]['std']=X_train[f].std()
            if bool_f == True:
                if (op == '>=' or op == '>'):
                    temp[f]['thr_2'] = max_f
                else:
                    temp[f]['thr_2'] = min_f
            bool_f=False
    df_viz = df_viz.from_dict(temp).T.reset_index().rename(columns={'index':'name'})
    inst = X_test.iloc[instance].values
    df_viz['inst'] = inst
    df_viz['rname']=df_viz['name']
    df_viz=df_viz.explode(['count'])
    df_viz = df_viz[((df_viz['count']!=l))].reset_index(drop=True)
    #df_viz=df_viz.sort_values(by='feature_importance',ascending=False)
   
    return df_viz

In [59]:
df=data_to_plot(instance=0)
df

,name,feature_importance,type,min,max,q1,median,q3,mean,std,op,thr,is_continuous,thr_2,count,inst,rname
0,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO,0.094664,numeric,0.0,1.0,0.42743,0.505918,0.559206,0.496342,0.167743,NaN,NaN,NaN,NaN,NaN,0.000000,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO
1,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO,0.081206,numeric,0.0,1.0,0.246011,0.366138,0.530745,0.410288,0.223558,NaN,NaN,NaN,NaN,NaN,0.227975,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO
2,PRODV_LETTERE_DI_CREDITO_PON,0.285429,numeric,0.0,1.0,0.452953,0.452953,0.452953,0.500201,0.14717,NaN,NaN,NaN,NaN,NaN,0.990218,PRODV_LETTERE_DI_CREDITO_PON
3,SCADV_FLG_RATA_DIVISA_SEK,0.0,numeric,0.0,0.0,0.0,0.0,0.0,0.0,0.0,=,1,False,0.0,NaN,0.000000,SCADV_FLG_RATA_DIVISA_SEK
4,OWNER_PRODV_FOREX_PON,0.334717,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 1.0]",1.000000,OWNER_PRODV_FOREX_PON
5,OWNER_PRODV_FOREX_PON,0.334717,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[3800, 358]",1.000000,OWNER_PRODV_FOREX_PON
6,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,0.115931,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 1.0]",1.000000,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON
7,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,0.115931,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[3975, 183]",1.000000,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON
8,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM,0.037198,numeric,0.0,1.0,0.174504,0.333834,0.634134,0.406126,0.285385,NaN,NaN,NaN,NaN,NaN,0.947305,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM
9,PN_LC_IMPORT_FLG_ONLY_TY,0.0,numeric,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,PN_LC_IMPORT_FLG_ONLY_TY


In [27]:
df['op'].unique()

array([nan, '='], dtype=object)

# plot function

In [28]:
def plot_rules(dataframe, only_rules=False):   
    def single_rule_plot_qualit(dataframe, rw):
        name= rw['name'].split('=')[0]
        data = dataframe[dataframe['rname'] == name]

        base= alt.Chart(
            data
        ).transform_stack(
            stack='count',
            as_=['count_start','count_end'],
            groupby=['rname'],
            sort=[alt.SortField('count', 'descending')]
        ).transform_calculate(
            midStack='(datum.count_start+datum.count_end)/2'
        )


        bar = base.mark_bar(
            stroke='white',
            color='lightgrey'
        ).encode(
            x='count_start:Q',
            x2='count_end:Q',
            tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
        )

        bar_r = base.mark_bar(
             stroke='#fcc40f'
         ).encode(
            x='count_start:Q',
            x2='count_end:Q',
            color=alt.condition('datum.is_continuous && datum.inst==1',alt.value("#fcc40f"),alt.value('white')),
            opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.0001)),
            tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
        )

        r=base.mark_bar(
            stroke='white'
        ).encode(
            x=alt.X(
                field='count',
                type='quantitative',
                title=None,
            ),
            y=alt.Y(
                field='rname',
                type='nominal',
                axis=None
            ),
            detail='name:N',
            color=alt.condition('datum.is_continuous',alt.value('#fcc40f'),alt.value('white')),
            opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
            tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
        )

        dot =base.mark_point(
            size=70,
            shape='diamond',
            color='black',
            filled=True
        ).encode(
            x=alt.X(
                field='midStack',
                type='quantitative',
                title=None,
            ),
            y=alt.Y(
                field='rname',
                type='nominal',
                axis=None
            ),
            opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
        )


        return alt.layer(bar,bar_r,dot).properties(
            height=20,
            width=300,
        )

    ########
    def single_index_text(dataframe, rw):
        data = dataframe[dataframe['name'] == rw['name']]
        chart = alt.Chart(
            data
        ).transform_calculate(
            label ="datum.type=='categorical' ? datum.name : datum.name +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
        ).mark_text(
            color='black',
            align='left',
            dx=-50,
            fontSize=13
        ).encode(
                text=alt.Text(
                field='label',
                type='nominal',
                title=None
            )
        )
        return chart.properties(
            height=20,
            width=101
        )
    ########
    def single_rule_plot_numeric(dataframe,rw):
        data = dataframe[dataframe['name'] == rw['name']]
        p=alt.Chart(
            data
        ).mark_point(
            color='black' if rw['is_continuous'] == True else 'black',
            size=70,
            shape='diamond',
            filled=True
        ).encode(
            x=alt.X(
                field='inst',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            tooltip=[alt.Tooltip(field='inst', title=rw['name'])]
        )

        t_min = alt.Chart(
            data
        ).mark_text(
            color='black',
            dx=-10,
            align='right'
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None
            ),
            text='min:N'
        )

        t_max = alt.Chart(
            data
        ).mark_text(
            color='black',
            dx=10,
            align='left'
        ).encode(
            x=alt.X(
                field='max',
                type='quantitative',
                title=None
            ),
            text='max:N'
        )



        b =alt.Chart(
            data
        ).mark_bar(
            color='#fcc40f',size=5,
            stroke='white'
        ).encode(
            x=alt.X(
                field='thr',
                type='quantitative',
                title=None,
            ),
            x2='thr2',
            y=alt.Y(
                field='name',
                type='nominal',
                title=None
            ),

        )



        l =alt.Chart(
            data
        ).mark_bar(
            color='grey',size=1
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            x2='max',
            y=alt.Y(field='name',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
        )


        q1_m = alt.Chart(
            data
        ).mark_bar(
            stroke='white',
            color='lightgrey',
            size=18
        ).encode(
            x=alt.X(
                field='q1',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            x2 = alt.X2(
                field='median'
            ),
        )

        m_q3 = alt.Chart(
            data
        ).mark_bar(
            stroke='white',
            color='lightgrey',
            size=18,
        ).encode(
            x=alt.X(
                field='median',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            x2 = alt.X2(
                field='q3'
            )
        )

        return alt.layer(l,q1_m,m_q3,b,p).properties(
            height=20,
            width=300        
        )
    ########
    def single_feature_importance_plot(dataframe, rw):
        data = dataframe[dataframe['name'] == rw['name']]
        chart = alt.Chart(
            data
        ).mark_bar(
        ).encode(
            x=alt.X(
                field='feature_importance',
                type='quantitative',
                title=None
            ),
            y=alt.Y(
                field='name',
                type='nominal',
                title=None,
                axis=None
            ),
            color=alt.condition('datum.feature_importance > 0', alt.value('#285588'), alt.value('#E36273')),
            tooltip=[alt.Tooltip(field="name"),alt.Tooltip(field="feature_importance")]
        )
        return chart.properties(
            height=20,
            width=100
        )
    ########
    
    ti_list=[]
    rp_list=[]
    fi_list=[]
    
    for i, row in dataframe.iterrows():
        if ((only_rules == True) and (row['is_continuous']!= True)):
            pass
        else:
            sti = single_index_text(dataframe, row)
            if row['type']== 'numeric':
                srp = single_rule_plot_numeric(dataframe, row)
            else:
                srp = single_rule_plot_qualit(dataframe, row)
            sfi = single_feature_importance_plot(dataframe, row)
            ti_list.append(sti)
            rp_list.append(srp)
            fi_list.append(sfi)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI').resolve_scale(
    x='shared'
)
    final_chart = alt.hconcat(
        fi_concat, rp_concat, ti_concat
    )

    return final_chart.configure(
       # background='#F5F5F5',
        padding=20
    ).configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [29]:
plot_rules(df)

alt.HConcatChart(...)